In [1]:
from pyspark.sql import SparkSession
from delta.tables import DeltaTable
!pwd
# batch_table_path = "/opt/consumer-output-delta-lake/delta-trades-batch-table"
# stream_table_path = "/opt/consumer-output-delta-lake/delta-trades-stream-5s-table"
checkpoint_table_path = "/opt/consumer-output-delta-lake/delta-trades-stream-5s-table/_delta_log/00000000000000004610.checkpoint.parquet"

batch_table_path = "/opt/consumer-output-iceberg/delta-trades-batch-table"
stream_table_path = "/opt/consumer-output-iceberg/delta-trades-stream-5s-table"

/opt/workspace


In [3]:
# import fastavro

# avro_path = "/opt/consumer-output-iceberg/crypto/trades/metadata/snap-8874187349782411031-1-de021b67-9317-496a-b141-2f8dcb7daee8.avro"

# with open(avro_path, 'rb') as f:
#     reader = fastavro.reader(f)
#     schema = reader.schema  # optional: print schema
#     records = list(reader)

# record_lst = []
# record_cnt = 0

# for record in records:
#     record_lst.append(record)
#     # record_cnt += record['data_file']['record_count']
#     print(record)

# len(record_lst)

## 1. Table description

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("DeltaInspect") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

# 1️⃣ ✅ SCHEMA + PARTITIONING + LOCATION + etc.
print("=== TABLE STRUCTURE (DESCRIBE DETAIL) ===")
spark.sql(f"DESCRIBE DETAIL delta.`{stream_table_path}`").show(truncate=True)

# 2️⃣ ✅ WRITE HISTORY (includes file count, row count, size)
print("\n=== LAST WRITE METRICS (DESCRIBE HISTORY) ===")
spark.sql(f"DESCRIBE HISTORY delta.`{stream_table_path}`").select(
    "version", "timestamp", "operation", "operationMetrics"
).show(truncate=False)

# 3️⃣ ✅ RAW SCHEMA (if you just want column names/types)
print("\n=== SCHEMA ONLY ===")
spark.sql(f"DESCRIBE delta.`{stream_table_path}`").show(30)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/01 14:31:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


=== TABLE STRUCTURE (DESCRIBE DETAIL) ===


AnalysisException: [DELTA_MISSING_DELTA_TABLE] `/opt/consumer-output-iceberg/delta-trades-stream-5s-table` is not a Delta table.

## 2. Table query

### 2.1. Batch table query

In [25]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("DeltaRead") \
    .config("spark.ui.port", "4041") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.sql.catalogImplementation", "in-memory") \
    .getOrCreate()

# Now this should work in OSS Delta Lake 3.2.0 + Spark 3.5.1
files_df = spark.sql(f"SELECT * FROM delta.`{batch_table_path}`")
print('Partitions from reading delta lake: ', files_df.rdd.getNumPartitions())
files_df.createOrReplaceTempView("batch_trades")
# files_df.show(5, truncate=True)

26/01/04 15:59:39 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


AnalysisException: [PATH_NOT_FOUND] Path does not exist: /opt/consumer-output-iceberg/delta-trades-batch-table.

In [6]:
df_batch_check = spark.sql(f"""
    SELECT
    exchange_minute, exchange_hour, count(distinct trade_id) as trade_cnt
    FROM batch_trades
    where exchange_day = 29
    group by 1, 2
    order by 2 desc, 1 desc
""").show(20)

### 2.2. Stream table query

#### 2.2.1. Read stream table

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("IcebergRead") \
    .config("spark.ui.port", "4043") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension,org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.sql.catalog.iceberg", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.iceberg.type", "hadoop") \
    .config("spark.sql.catalog.iceberg.warehouse", "/opt/consumer-output-iceberg") \
    .getOrCreate()

spark.sql(f"""
    SELECT
        symbol,
        max(exchange_time_ts) as candle_time,
        first(price) as open_price,
        max(price) as high_price,
        min(price) as low_price,
        last(price) as close_price,
        sum(size) as volume,
        count(distinct trade_id) as trade_count
    FROM iceberg.crypto.trades_stream
    GROUP BY symbol, window(exchange_time_ts, '5 minutes')
    ORDER BY candle_time desc
""").show(truncate=False)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/05 07:03:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/01/05 07:03:52 WARN FileSystem: Cannot load filesystem: java.util.ServiceConfigurationError: org.apache.hadoop.fs.FileSystem: com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem Unable to get public no-arg constructor
26/01/05 07:03:52 WARN FileSystem: java.lang.NoClassDefFoundError: com/google/api/client/http/HttpRequestInitializer
26/01/05 07:03:52 WARN FileSystem: java.lang.ClassNotFoundException: com.google.api.client.http.HttpRequestInitializer
                                                                                

+-------+--------------------------+----------+----------+---------+-----------+------------------+-----------+
|symbol |candle_time               |open_price|high_price|low_price|close_price|volume            |trade_count|
+-------+--------------------------+----------+----------+---------+-----------+------------------+-----------+
|BTC-USD|2026-01-05 14:03:49.585415|92540.0   |92544.0   |92428.01 |92428.02   |8.67600211        |1021       |
|XRP-USD|2026-01-05 14:03:49.210659|2.1295    |2.1298    |2.1224   |2.1225     |90502.055627      |320        |
|ETH-USD|2026-01-05 14:03:49.193611|3159.62   |3160.38   |3154.21  |3155.88    |68.8263445        |661        |
|SOL-USD|2026-01-05 14:03:49.16037 |135.57    |135.68    |135.41   |135.46     |2883.0628209799997|516        |
|ADA-USD|2026-01-05 14:03:37.898954|0.4011    |0.4012    |0.3987   |0.3994     |213642.87711815   |247        |
|BTC-USD|2026-01-05 13:59:59.344899|92454.44  |92466.5   |92418.87 |92436.01   |6.7558878         |981  

In [ ]:
from datetime import datetime, timedelta
import pytz

cutoff_timestamp_ms = int((datetime.now(pytz.utc) - timedelta(minutes=90)).timestamp() * 1000)

spark.sql(f"""
    CALL iceberg.system.expire_snapshots(
        table => 'iceberg.crypto.trades_stream',
        older_than => {cutoff_timestamp_ms},
    )
""").show()

#### 2.2.2. Checking data quality

In [6]:
spark.sql(f"""
    SELECT
        symbol,
        max(exchange_time_ts) as candle_time,
        first(price) as open_price,
        max(price) as high_price,
        min(price) as low_price,
        last(price) as close_price,
        sum(size) as volume,
        count(distinct trade_id) as trade_count
    FROM stream_trades
    GROUP BY symbol, window(exchange_time_ts, '5 minutes')
    ORDER BY symbol, candle_time desc
""").show(truncate=False)

+-------+--------------------------+----------+----------+---------+-----------+------------------+-----------+
|symbol |candle_time               |open_price|high_price|low_price|close_price|volume            |trade_count|
+-------+--------------------------+----------+----------+---------+-----------+------------------+-----------+
|ADA-USD|2026-01-03 23:11:10.919916|0.386     |0.386     |0.3854   |0.3857     |18852.389270959997|37         |
|ADA-USD|2026-01-03 23:09:50.27356 |0.3856    |0.3872    |0.3855   |0.3858     |38698.06143051004 |149        |
|BTC-USD|2026-01-03 23:11:18.325865|89941.84  |89959.19  |89936.0  |89936.0    |0.6758629200000001|166        |
|BTC-USD|2026-01-03 23:09:59.563874|89929.02  |89997.34  |89923.0  |89938.86   |4.848338999999998 |774        |
|ETH-USD|2026-01-03 23:11:14.504647|3100.43   |3101.79   |3100.16  |3101.28    |11.466451069999998|118        |
|ETH-USD|2026-01-03 23:09:59.139666|3102.96   |3104.07   |3099.75  |3100.17    |77.31306644000001 |522  

#### 2.2.3. Data transformation

In [24]:
df_5min = spark.sql("""
    SELECT
        window.start AS window_start,
        window.end AS window_end,
        symbol,
        COUNT(*) AS trade_count,
        SUM(case when side = 'buy' then size end) AS volume_buy,
        SUM(case when side = 'sell' then size end) AS volume_sell,
        AVG(price) AS avg_price,
        MIN(price) AS min_price,
        MAX(price) AS max_price,
        LAST(price, TRUE) AS close_price,  -- last trade price in window
        FIRST(price, TRUE) AS open_price   -- first trade price in window
    FROM (
        SELECT *,
               window(exchange_time_ts, '5 minutes') AS window
        FROM stream_trades
    )
    GROUP BY window, symbol
    ORDER BY window_start DESC, symbol
""")

df_5min.printSchema()
df_5min.show(10, truncate=False)

root
 |-- window_start: timestamp (nullable = true)
 |-- window_end: timestamp (nullable = true)
 |-- symbol: string (nullable = true)
 |-- trade_count: long (nullable = false)
 |-- volume_buy: double (nullable = true)
 |-- volume_sell: double (nullable = true)
 |-- avg_price: double (nullable = true)
 |-- min_price: double (nullable = true)
 |-- max_price: double (nullable = true)
 |-- close_price: double (nullable = true)
 |-- open_price: double (nullable = true)



[Stage 154:==================================================>    (37 + 3) / 40]

+-------------------+-------------------+-------+-----------+------------------+------------------+-------------------+---------+---------+-----------+----------+
|window_start       |window_end         |symbol |trade_count|volume_buy        |volume_sell       |avg_price          |min_price|max_price|close_price|open_price|
+-------------------+-------------------+-------+-----------+------------------+------------------+-------------------+---------+---------+-----------+----------+
|2025-12-29 16:45:00|2025-12-29 16:50:00|ADA-USD|364        |162364.71059076997|109395.70658647   |0.36769670329670345|0.3669   |0.3687   |0.3676     |0.3673    |
|2025-12-29 16:45:00|2025-12-29 16:50:00|BTC-USD|3482       |40.28869031000001 |10.846198720000007|87960.60836588149  |87869.59 |88134.36 |87921.37   |88020.0   |
|2025-12-29 16:45:00|2025-12-29 16:50:00|ETH-USD|1360       |210.04045015000008|170.71252654000006|2959.5825882352938 |2953.72  |2967.72  |2959.32    |2963.21   |
|2025-12-29 16:45:00|2

## 3. Time travel

In [11]:
# Read the table as it existed at a specific time
df_past = spark.sql(f"""
    SELECT * FROM delta.`{table_path}`
    TIMESTAMP AS OF '2025-12-28 19:00:00'
""")
df_past.count()

1724793

In [14]:
# Read the table as it existed at a specific time
df_current = spark.sql(f"""
    SELECT * FROM delta.`{table_path}`
""")
df_current.count()

2298468